# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [3]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

Python 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
asyncio version: built-in


## Sección 1: await secuencial vs asyncio.gather — la diferencia central

La diferencia entre M2 (await secuencial) y M4 (gather) es una línea de código.
Medir los tiempos hace la diferencia completamente visible.

In [4]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

=== M2: await secuencial ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M2: 5.00s  (esperado: 5.0s = N×T)



In [5]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

=== M4: asyncio.gather ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M4: 1.00s  (esperado: ~1.0s = T_max)

Speedup M4/M2: 5.0x

Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅
Las 5 tareas de 1.0s corren en ~1.0s en lugar de 5.0s


## Sección 2: Traza del event loop con asyncio debug mode

asyncio tiene un modo de depuración que muestra advertencias cuando el event loop se bloquea.
Aquí vemos la diferencia entre `asyncio.sleep` y `time.sleep`.

In [6]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

=== gather con tareas CORRECTAS (asyncio.sleep) ===
  A: inicio
  B: inicio
  C: inicio
  A: fin
  B: fin
  C: fin
Tiempo: 0.31s  (esperado: ~0.3s)

=== gather con tareas BLOQUEANTES (time.sleep) ===
  X: inicio
  X: fin
  Y: inicio
  Y: fin
  Z: inicio
  Z: fin
Tiempo: 0.91s  (esperado: ~0.9s — sin mejora)

Observa: con time.sleep, gather NO ayuda.
time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.


La diferencia se ve en que en el primero si pudo aprovechar las esperas y en el segundo no.

In [7]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

Implementa los modelos M2 y M3 explícitamente y mide por qué no producen mejora sobre M1 para sus respectivos casos.

In [8]:
import time
import threading

# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
# Responde con la definición formal: ¿qué condición de M4 falta en M2?
print("M2 no mejora sobre M1 porque, aunque usa async/await, las tareas se esperan una por una.")
print("Formalmente, en M2 falta concurrencia: no hay múltiples tareas activas al mismo tiempo")
print("(como sí ocurre en M4 con gather/create_task). Por eso no se solapan los tiempos de espera.\n")


# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(t_i) = 0"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# --- Threading M3 ---
hilos = []

t0 = time.perf_counter()
for _ in range(N_HILOS):
    h = threading.Thread(target=tarea_cpu_bound, args=(N_CPU,))
    hilos.append(h)
    h.start()

for h in hilos:
    h.join()

t_threading = time.perf_counter() - t0

# --- Resultados ---
speedup = t_secuencial / t_threading

print(f'M1 secuencial: {t_secuencial:.2f}s')
print(f'M3 threading:  {t_threading:.2f}s')
print(f'Speedup M3/M1: {speedup:.2f}x')

if speedup > 1.05:
    print("Interpretación: hubo una mejora pequeña, pero no es la esperada de paralelismo real.")
elif speedup < 0.95:
    print("Interpretación: hubo slowdown, consistente con overhead + GIL.")
else:
    print("Interpretación: prácticamente no hubo mejora, consistente con el GIL.")

print("\nConclusión:")
print("M3 no mejora para CPU-bound porque los hilos en CPython comparten el GIL,")
print("así que no ejecutan bytecode Python en paralelo real. Como además hay overhead")
print("de creación/coordinación de hilos, el resultado suele ser ~igual o incluso peor que secuencial.")

M2 no mejora sobre M1 porque, aunque usa async/await, las tareas se esperan una por una.
Formalmente, en M2 falta concurrencia: no hay múltiples tareas activas al mismo tiempo
(como sí ocurre en M4 con gather/create_task). Por eso no se solapan los tiempos de espera.

M1 secuencial: 1.95s
M3 threading:  1.85s
Speedup M3/M1: 1.06x
Interpretación: hubo una mejora pequeña, pero no es la esperada de paralelismo real.

Conclusión:
M3 no mejora para CPU-bound porque los hilos en CPython comparten el GIL,
así que no ejecutan bytecode Python en paralelo real. Como además hay overhead
de creación/coordinación de hilos, el resultado suele ser ~igual o incluso peor que secuencial.


## Sección 4: Race condition reproducible + fix con Lock

Las condiciones de carrera son consecuencia de la memoria compartida en concurrencia.
Reproduce el problema y aplica la solución con `threading.Lock`.

In [10]:
import threading

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1   # NO atómico: LOAD, ADD, STORE separados

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Sin lock — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print()

# TAREA 4.2 — Fix con Lock
lock = threading.Lock()
contador_con_lock = [0]

def incrementar_con_lock():
    for _ in range(N_INCREMENTOS):
        with lock:
            contador_con_lock[0] += 1

hilos = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print(f'Con lock — esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}')
print(f'¿contador_con_lock == esperado? {contador_con_lock[0] == esperado}')

Sin lock — esperado: 400,000, obtenido: 400,000
Diferencia: 0 incrementos perdidos

Con lock — esperado: 400,000, obtenido: 400,000
¿contador_con_lock == esperado? True


No se si hice algo mal pero me dio igual en ambos casos. No se si es porque python ya tiene implementado algo que evita los race conditions.

## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

Implementa el servidor chatbot v2 usando asyncio y mide su comportamiento con N usuarios.

In [12]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(t_i): I/O a base de datos ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(t_i): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    t_inicio = time.perf_counter()

    historial = await consultar_bd(user_id)
    respuesta = await llamar_llm(historial)

    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# TAREA 5.1 — Servidor secuencial (chatbot v1 como baseline)
async def servidor_v1(n_usuarios: int):
    """M1: un usuario a la vez"""
    resultados = []
    t0 = time.perf_counter()

    for i in range(n_usuarios):
        r = await handle_request(i)
        resultados.append(r)

    t_total = time.perf_counter() - t0
    return resultados, t_total

# TAREA 5.2 — Servidor concurrente (chatbot v2)
async def servidor_v2(n_usuarios: int):
    """M4: todos los usuarios concurrentes con gather"""
    t0 = time.perf_counter()

    tareas = [handle_request(i) for i in range(n_usuarios)]
    resultados = await asyncio.gather(*tareas)

    t_total = time.perf_counter() - t0
    return resultados, t_total

# TAREA 5.3 — Compara v1 vs v2 con N=10 usuarios
N = 10
print(f'Comparando v1 vs v2 con {N} usuarios...\n')

resultados_v1, t_total_v1 = await servidor_v1(N)
lat_prom_v1 = sum(r['latencia'] for r in resultados_v1) / N

resultados_v2, t_total_v2 = await servidor_v2(N)
lat_prom_v2 = sum(r['latencia'] for r in resultados_v2) / N

speedup = t_total_v1 / t_total_v2

print("=== Resultados ===")
print(f'v1 secuencial -> tiempo total: {t_total_v1:.2f}s | latencia promedio: {lat_prom_v1:.2f}s')
print(f'v2 concurrente -> tiempo total: {t_total_v2:.2f}s | latencia promedio: {lat_prom_v2:.2f}s')
print(f'Speedup v2/v1: {speedup:.2f}x\n')

print("=== Interpretación ===")
print("v2 suele ser mucho más rápido en tiempo total porque solapa las esperas de I/O")
print("(base de datos + llamada al LLM) entre múltiples usuarios.")
print("La latencia de cada usuario en v2 suele ser similar a la de v1 porque cada petición")
print("sigue teniendo que esperar su propia BD y su propia llamada al LLM; lo que mejora")
print("es el throughput total del servidor, no necesariamente la latencia individual.\n")

print("=== Latencias individuales v2 ===")
for r in resultados_v2:
    print(f'usuario {r["user"]}: {r["latencia"]:.2f}s')

Comparando v1 vs v2 con 10 usuarios...

=== Resultados ===
v1 secuencial -> tiempo total: 14.97s | latencia promedio: 1.50s
v2 concurrente -> tiempo total: 2.00s | latencia promedio: 1.72s
Speedup v2/v1: 7.48x

=== Interpretación ===
v2 suele ser mucho más rápido en tiempo total porque solapa las esperas de I/O
(base de datos + llamada al LLM) entre múltiples usuarios.
La latencia de cada usuario en v2 suele ser similar a la de v1 porque cada petición
sigue teniendo que esperar su propia BD y su propia llamada al LLM; lo que mejora
es el throughput total del servidor, no necesariamente la latencia individual.

=== Latencias individuales v2 ===
usuario 0: 1.80s
usuario 1: 1.20s
usuario 2: 1.54s
usuario 3: 1.90s
usuario 4: 1.61s
usuario 5: 1.85s
usuario 6: 2.00s
usuario 7: 1.74s
usuario 8: 1.99s
usuario 9: 1.54s
